# Merged 14-Class ECG Classifier (PTB-XL + Chapman)

Trains ECGNetJoint on **60,914 records** (18,524 PTB-XL + 42,390 Chapman-Shaoxing).
Adds **AFIB** (48 → 9,840 samples) and **STACH** (4 → 7,255 samples) to the 12-class model.

---
## Runtime guide

| Cell | Runtime | Duration | What it does |
|------|---------|----------|--------------|
| 1 | **CPU** | 1 min | Mount Drive + install deps |
| 2 | **CPU** | 1 min | Copy scripts from Drive |
| 3 | **CPU** | ~1.5 hrs | Download Chapman from PhysioNet |
| 4 | **CPU** | ~10 min | Build Chapman index + save tar to Drive |
| 5 | **GPU** | 1 min | Switch to GPU — check + mount Drive |
| 6 | **GPU** | ~20 min | Restore Chapman + copy PTB-XL to SSD |
| 7 | **GPU** | ~1-2 hrs | Run merged training |
| 8 | **GPU** | 1 min | Save model to Drive |

**Before running:** Upload these 4 scripts to `MyDrive/EKG_models/`:
`multilabel_merged.py`, `multilabel_classifier.py`, `cnn_classifier.py`, `dataset_chapman.py`

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1  ──  CPU runtime
# Mount Drive and install dependencies
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import subprocess, sys
print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'wfdb', 'scipy', 'scikit-learn'], check=True)

import torch
print(f'Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}  (expected False on CPU runtime)')
print('Cell 1 done')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2  ──  CPU runtime
# Copy training scripts from Drive to /content/
# ─────────────────────────────────────────────────────────────────────────────
import os, shutil

SCRIPTS_DIR = '/content/drive/MyDrive/EKG_models'
NEEDED = [
    'multilabel_merged.py',
    'multilabel_classifier.py',
    'cnn_classifier.py',
    'dataset_chapman.py',
]

os.chdir('/content')
os.makedirs('ekg_datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)

missing = []
for script in NEEDED:
    src = f'{SCRIPTS_DIR}/{script}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/{script}')
        print(f'  Copied {script}')
    else:
        missing.append(script)

if missing:
    raise FileNotFoundError(
        f'Missing scripts on Drive: {missing}\n'
        f'Upload them to {SCRIPTS_DIR}/ first.'
    )
print('Cell 2 done - all scripts copied')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3  ──  CPU runtime
# Download Chapman ZIP from PhysioNet (~2.3 GB) then extract
# Single file download — much faster than 90K individual files
# Expected: 10-20 min on Colab's fast connection
# Safe to re-run: skips if already extracted
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess, shutil
from pathlib import Path

CHAPMAN_DIR = '/content/ekg_datasets/chapman'
LOCAL_ZIP   = '/content/chapman.zip'
DRIVE_ZIP   = '/content/drive/MyDrive/EKG_models/chapman.zip'
ZIP_URL     = ('https://physionet.org/static/published-projects/ecg-arrhythmia/'
               'a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0.zip')

os.makedirs(CHAPMAN_DIR, exist_ok=True)

n_existing = len(list(Path(CHAPMAN_DIR).rglob('*.mat')))
if n_existing >= 44000:
    print(f'Chapman already extracted: {n_existing} .mat files - skipping')
else:
    # Option A: use ZIP already on Drive (if you uploaded it manually)
    if os.path.exists(DRIVE_ZIP):
        size_gb = os.path.getsize(DRIVE_ZIP) / 1e9
        print(f'Found chapman.zip on Drive ({size_gb:.1f} GB) — copying to SSD...')
        shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
    else:
        # Option B: download directly from PhysioNet (~2.3 GB, 10-20 min on Colab)
        print('Downloading Chapman ZIP from PhysioNet (~2.3 GB)...')
        result = subprocess.run([
            'wget', '-c', '-q', '--show-progress',
            '-O', LOCAL_ZIP, ZIP_URL
        ], check=False)
        if result.returncode != 0 or not os.path.exists(LOCAL_ZIP):
            raise RuntimeError('Download failed. Check your internet connection or upload chapman.zip to EKG_models/ manually.')
        size_gb = os.path.getsize(LOCAL_ZIP) / 1e9
        print(f'Downloaded: {size_gb:.1f} GB')

    print('Extracting ZIP...')
    subprocess.run(['unzip', '-q', LOCAL_ZIP, '-d', '/content/chapman_extract'], check=True)

    extract_base = Path('/content/chapman_extract')
    wfdb_matches = list(extract_base.rglob('WFDBRecords'))
    if wfdb_matches:
        dest = Path(CHAPMAN_DIR) / 'WFDBRecords'
        if dest.exists():
            shutil.rmtree(dest)
        shutil.move(str(wfdb_matches[0]), str(dest))
    else:
        for item in extract_base.iterdir():
            shutil.move(str(item), CHAPMAN_DIR)
    shutil.rmtree('/content/chapman_extract', ignore_errors=True)
    os.remove(LOCAL_ZIP)

n_after = len(list(Path(CHAPMAN_DIR).rglob('*.mat')))
print(f'Done: {n_after} .mat files in {CHAPMAN_DIR}')
if n_after < 44000:
    print(f'WARNING: only {n_after}/45152 files — re-run this cell to retry')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4  ──  CPU runtime
# Build Chapman index CSV (needed for training)
# ~5-10 min
# ─────────────────────────────────────────────────────────────────────────────
import os
os.chdir('/content')

INDEX_PATH = '/content/ekg_datasets/chapman_index.csv'
if os.path.exists(INDEX_PATH):
    import pandas as pd
    n = len(pd.read_csv(INDEX_PATH))
    print(f'Index already exists: {n} records - skipping')
else:
    print('Building Chapman index (~5-10 min)...')
    import dataset_chapman
    dataset_chapman.build_chapman_index(
        '/content/ekg_datasets/chapman',
        INDEX_PATH
    )
    import pandas as pd
    n = len(pd.read_csv(INDEX_PATH))
    print(f'Index built: {n} records -> {INDEX_PATH}')

# Save index to Drive so GPU session can use it
DRIVE_MODELS = '/content/drive/MyDrive/EKG_models'
import shutil
os.makedirs(DRIVE_MODELS, exist_ok=True)
drive_index = f'{DRIVE_MODELS}/chapman_index.csv'
shutil.copy(INDEX_PATH, drive_index)
print(f'Index saved to Drive: {drive_index}')

print('\nCell 4 done.')
print('>>> NEXT STEP: Switch to GPU runtime (Runtime > Change runtime type > T4 GPU > Save)')
print('>>> After restart, run Cells 5, 6, 7, 8')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5  ──  GPU runtime  (run FIRST after switching to GPU)
# Verify GPU, mount Drive, install dependencies, copy scripts
# ─────────────────────────────────────────────────────────────────────────────
import torch, sys
if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected!\n'
        'Go to Runtime > Change runtime type > T4 GPU > Save'
    )
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'wfdb', 'scipy', 'scikit-learn'], check=True)

import os, shutil
SCRIPTS_DIR = '/content/drive/MyDrive/EKG_models'
os.chdir('/content')
os.makedirs('ekg_datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)

for script in ['multilabel_merged.py', 'multilabel_classifier.py',
               'cnn_classifier.py', 'dataset_chapman.py']:
    shutil.copy(f'{SCRIPTS_DIR}/{script}', f'/content/{script}')
    print(f'  Copied {script}')

print('Cell 5 done')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6  ──  GPU runtime
# Restore Chapman from Drive tar + copy PTB-XL to local SSD
# Chapman restore: ~5 min (tar extract)
# PTB-XL copy:    ~10-15 min
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess, shutil
from pathlib import Path
os.chdir('/content')

DRIVE_MODELS  = '/content/drive/MyDrive/EKG_models'
CHAPMAN_DIR   = '/content/ekg_datasets/chapman'
CHAPMAN_INDEX = '/content/ekg_datasets/chapman_index.csv'
os.makedirs(CHAPMAN_DIR, exist_ok=True)

# ── Restore Chapman files from tar ───────────────────────────────────────────
n_existing = len(list(Path(CHAPMAN_DIR).rglob('*.mat')))
if n_existing >= 44000:
    print(f'Chapman already present: {n_existing} .mat files - skipping')
else:
    DRIVE_TAR = f'{DRIVE_MODELS}/chapman.tar'
    if not os.path.exists(DRIVE_TAR):
        raise FileNotFoundError(f'chapman.tar not found: {DRIVE_TAR}')
    size_gb = os.path.getsize(DRIVE_TAR) / 1e9
    print(f'Restoring Chapman from tar ({size_gb:.1f} GB)...')
    subprocess.run(['tar', 'xf', DRIVE_TAR, '-C', '/content'], check=True)
    n_after = len(list(Path(CHAPMAN_DIR).rglob('*.mat')))
    print(f'Chapman restored: {n_after} .mat files')

# ── Restore Chapman index CSV ─────────────────────────────────────────────────
if os.path.exists(CHAPMAN_INDEX):
    import pandas as pd
    print(f'Chapman index present: {len(pd.read_csv(CHAPMAN_INDEX))} records')
else:
    DRIVE_INDEX = f'{DRIVE_MODELS}/chapman_index.csv'
    if os.path.exists(DRIVE_INDEX):
        shutil.copy(DRIVE_INDEX, CHAPMAN_INDEX)
        import pandas as pd
        print(f'Chapman index restored from Drive: {len(pd.read_csv(CHAPMAN_INDEX))} records')
    else:
        print('Building Chapman index (~5-10 min)...')
        import dataset_chapman
        dataset_chapman.build_chapman_index(CHAPMAN_DIR, CHAPMAN_INDEX)
        import pandas as pd
        print(f'Chapman index built: {len(pd.read_csv(CHAPMAN_INDEX))} records')

# ── Copy PTB-XL to local SSD ──────────────────────────────────────────────────
PTBXL_LOCAL = '/content/ekg_datasets/ptbxl'
DRIVE_PTBXL = f'{DRIVE_MODELS}/ptbxl'

if not os.path.exists(DRIVE_PTBXL):
    raise FileNotFoundError(f'PTB-XL not found on Drive: {DRIVE_PTBXL}')

if os.path.exists(PTBXL_LOCAL):
    n_dat = len(list(Path(PTBXL_LOCAL).rglob('*.dat')))
    print(f'PTB-XL already on SSD: {n_dat} .dat files - skipping copy')
else:
    n_on_drive = len(list(Path(DRIVE_PTBXL).rglob('*.dat')))
    print(f'Copying PTB-XL from Drive to SSD ({n_on_drive} .dat files, ~10-15 min)...')
    shutil.copytree(DRIVE_PTBXL, PTBXL_LOCAL)
    n_dat = len(list(Path(PTBXL_LOCAL).rglob('*.dat')))
    print(f'PTB-XL copied: {n_dat} .dat files on SSD')

print('\nCell 6 done - data ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7  ──  GPU runtime
# Run merged 14-class training
# Expected: ~1-2 hrs on T4 (vs 15-25 hrs on CPU)
# Prints per-epoch: Loss | ValAUROC | ValF1
# Saves best model automatically to /content/models/ecg_multilabel_v2.pt
# ─────────────────────────────────────────────────────────────────────────────
import os, subprocess, sys
os.chdir('/content')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Run Cell 5 first or switch runtime to T4.')

result = subprocess.run(
    [sys.executable, '-u', 'multilabel_merged.py'],
    check=False
)

if result.returncode != 0:
    print(f'\nTraining exited with code {result.returncode}')
else:
    print('\nTraining complete!')
    model_path = '/content/models/ecg_multilabel_v2.pt'
    if os.path.exists(model_path):
        size_mb = os.path.getsize(model_path) / 1e6
        print(f'Model saved: {model_path} ({size_mb:.1f} MB)')
    else:
        print('WARNING: Model file not found at expected path')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8  ──  GPU runtime
# Save trained model to Drive
# ─────────────────────────────────────────────────────────────────────────────
import os, shutil

DRIVE_MODELS = '/content/drive/MyDrive/EKG_models'
os.makedirs(DRIVE_MODELS, exist_ok=True)

files_to_save = [
    ('/content/models/ecg_multilabel_v2.pt', f'{DRIVE_MODELS}/ecg_multilabel_v2.pt'),
]

for src, dst in files_to_save:
    if os.path.exists(src):
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'Saved: {dst} ({size_mb:.1f} MB)')
    else:
        print(f'WARNING: {src} not found - training may have failed')

print('\nDone! Download ecg_multilabel_v2.pt from Drive and place it in models/ locally.')